# Семинар 4. Запросы к одной таблице

На этом занятии вы будете писать запросы вида `SELECT … FROM … WHERE … ORDER BY … LIMIT …`
к демобазе авиаперевозок: фильтровать строки, вычислять новые столбцы, сортировать,
искать по шаблону и собирать текст.

**Как устроен ноутбук.** Каждая задача — описание и ячейка, которая начинается с `%%sql`.
Всё, что написано в такой ячейке ниже первой строки, отправляется в PostgreSQL,
а результат показывается таблицей.

**База.** PostgreSQL 17 в Docker, демобаза авиаперевозок Postgres Pro (версия 2016 года).
Как поднять базу, открыть ноутбук и подключиться через DBeaver, написано в файле `README.md`
в корне репозитория.

Все таблицы лежат в схеме `bookings`, в запросах пишем имя вместе со схемой: `bookings.flights`.

### Таблицы демобазы, которые понадобятся

| Таблица | Одна строка — это | Нужные столбцы |
|---|---|---|
| `bookings.airports` | аэропорт | `airport_code` — код из трёх букв, `airport_name`, `city` |
| `bookings.aircrafts` | модель самолёта | `aircraft_code`, `model`, `range` — дальность, км |
| `bookings.flights` | рейс в конкретный день | `flight_id`, `flight_no`, `departure_airport`, `arrival_airport`, `aircraft_code`, `status`, `scheduled_departure`, `scheduled_arrival` |
| `bookings.ticket_flights` | перелёт по билету на конкретном рейсе | `ticket_no`, `flight_id`, `fare_conditions` — класс (`Economy`, `Comfort`, `Business`), `amount` — цена |
| `bookings.tickets` | билет пассажира | `ticket_no`, `passenger_name`, `contact_data` — контакты (телефон и email) |

Статусы рейса: `Scheduled` — продажа открыта, `On Time`, `Delayed`, `Departed`, `Arrived`, `Cancelled`.
Данные сформированы на момент 13.10.2016 17:00 — у рейсов после этой даты статус `Scheduled`.

**Схема демобазы.** Таблицы, их ключи и связи; подробное описание столбцов — в [`db/schema.md`](../db/schema.md).

![Схема демобазы авиаперевозок](../db/schema.svg)

In [ ]:
import sys; sys.path.insert(0, "../jupyter")   # магия %%sql лежит в папке jupyter репозитория
%load_ext sql_magic

### Проверка подключения
Если ячейка ниже показала пять аэропортов — всё готово к работе.

In [ ]:
%%sql
SELECT * FROM bookings.airports LIMIT 5;

# Задачи: распродажа билетов на 20 октября 2016

Авиакомпания готовит распродажу билетов на рейсы 20 октября 2016 года из Москвы и Санкт-Петербурга.
Задачи идут цепочкой: результат одной нужен в следующей.

**Памятка**

```sql
SELECT столбец, выражение AS имя        -- что вывести
FROM bookings.таблица                     -- откуда
WHERE условие AND (условие OR условие)  -- какие строки оставить
ORDER BY столбец DESC                   -- как отсортировать
LIMIT 10;                               -- сколько строк вывести
```

| Что | Как пишется | Пример |
|---|---|---|
| Значение из списка | `IN (…)`, `NOT IN (…)` | `city IN ('Москва', 'Казань')` |
| Уникальные строки | `SELECT DISTINCT` | `SELECT DISTINCT status FROM bookings.flights` |
| Округление вниз | `floor(x)` | `floor(99.9)` → `99` |
| Шаблон | `LIKE`, `ILIKE`, `~` | `model LIKE 'Boeing%'`, `email ~ '\d{4}@'` |
| Строки | `\|\|`, `initcap`, `split_part` | `split_part('IVAN PETROV', ' ', 1)` → `IVAN` |
| Дата из момента | `значение::date` | `scheduled_departure::date = '2016-10-20'` |

### Задача 1. Коды аэропортов и самолётов для акции

В таблице рейсов аэропорты и самолёты указаны только кодами. Чтобы отобрать рейсы для акции,
сначала найдите нужные коды.

1. Аэропорты Москвы и Санкт-Петербурга. Выведите **airport_code**, **airport_name**, **city**.
2. Самолёты Boeing 777-300, Sukhoi SuperJet-100 и Bombardier CRJ-200. Выведите **aircraft_code**, **model**.

In [ ]:
%%sql
-- Задача 1.1. Напишите запрос

In [ ]:
%%sql
-- Задача 1.2. Напишите запрос

### Задача 2. Акционные рейсы

Акция действует на рейсы **20 октября 2016 года**, продажа на которые ещё открыта (статус `Scheduled`):

- **из Москвы** — все рейсы на Boeing 777-300, кроме вылетов из Внуково;
- **из Санкт-Петербурга** — рейсы на региональных самолётах Superjet и CRJ-200:
  на Superjet — если полёт по расписанию короче 1 ч 15 мин, на CRJ-200 — если короче 2 ч.

Коды аэропортов и самолётов возьмите из задачи 1. Длительность полёта —
`scheduled_arrival - scheduled_departure`, её можно сравнивать с `interval '1 hour 15 minutes'`.

Выведите **flight_id**, **flight_no**, **departure_airport**, **arrival_airport**, **aircraft_code**,
**duration** (длительность), отсортируйте по **flight_id**.

In [ ]:
%%sql
-- Задача 2. Напишите запрос

### Задача 3. Цены со скидкой 5% и 10%

Для акционных рейсов из задачи 2 решаем, какую скидку дать: 5% или 10%.
Посчитайте цены билетов **эконом-класса** (`fare_conditions = 'Economy'`) с каждой скидкой.
Округлите вниз до целого.

Цены лежат в `bookings.ticket_flights`: одна строка — один проданный перелёт, поэтому одна и та же
цена повторяется много раз. Выведите каждую цену один раз.

Выведите **flight_id**, **amount**, **price_5**, **price_10**, отсортируйте по **flight_id** и **amount**.

In [ ]:
%%sql
-- Задача 3. Напишите запрос

### Задача 4. Минимальная цена со скидкой

Решено давать скидку 10%. Найдите самую низкую цену эконома со скидкой 10% (с округлением вниз):

1. среди акционных рейсов **из Москвы**;
2. среди акционных рейсов **из Санкт-Петербурга**.

Выведите **flight_id**, **amount**, **price_10**.

In [ ]:
%%sql
-- Задача 4.1. Напишите запрос

In [ ]:
%%sql
-- Задача 4.2. Напишите запрос

### Задача 5. Пассажиры для рассылки

Рассылку отправим по почте тем пассажирам, у которых в адресе почты перед `@` стоит полная дата
рождения — **8 цифр подряд** (например, `volkova.alina_03101973@…`).

Контакты пассажира лежат в столбце `contact_data`, адрес почты достаётся так:
`contact_data->>'email'`, телефон — `contact_data->>'phone'`.

Выведите **ticket_no**, **passenger_name**, **email**, **phone** — первые 20 строк по номеру билета.

In [ ]:
%%sql
-- Задача 5. Напишите запрос

### Задача 6. Текст рассылки

Подготовим письма пассажирам из задачи 5. Текст по шаблону:

«{Имя}, добрый день! Распродажа билетов на 20 октября: из Москвы от {X} рублей, из Петербурга — от {Y}!»

Вместо X и Y подставьте цены из задачи 4. Имя в `passenger_name` записано заглавными
вместе с фамилией: возьмите только имя и приведите его к виду «Alina».

Выведите **ticket_no**, **email**, **message_text** — первые 20 строк по номеру билета.

In [ ]:
%%sql
-- Задача 6. Напишите запрос